<h1 style="color:#2E86AB; font-family:Georgia; text-align:center; border-bottom: 3px solid #2E86AB; padding-bottom:10px;">
🔗 Merge y Feature Engineering
</h1>

**Autor:** Eddy Luis  
**Fecha:** Mayo 2026  
**Herramientas:** Python · Pandas · NumPy

---

## 📌 Descripción
Unimos el dataset de **centros educativos** con los **indicadores de abandono, promoción y reprobación** extraídos de los PDFs del MINERD. Luego creamos las variables (features) que alimentarán el modelo predictivo de deserción escolar.

---

## 📋 Contenido de este Notebook

1. 📦 Importar Librerías y Cargar Datos
2. 🧹 Armonizar Nombres de Distritos
3. 🔗 Merge: Centros + Indicadores
4. 🏷️ Crear Flags de Niveles Ofrecidos
5. 📐 Asignar Tasa de Abandono Relevante
6. 📊 Features a Nivel de Distrito
7. 📈 Tendencia del Abandono (2021→2025)
8. ⚠️ Clasificación de Riesgo
9. 💾 Guardar Dataset Final
10. 🔍 Resumen del Dataset

---

<h2 style="color:#A23B72; font-family:Georgia;">
📦 1. Importar Librerías y Cargar Datos
</h2>

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")

df_centros = pd.read_csv(PROCESSED_DIR / "centros_educativos_limpios.csv")
df_indicadores = pd.read_csv(PROCESSED_DIR / "indicadores_educativos_ancho.csv")

print(f"Centros educativos: {df_centros.shape}")
print(f"Indicadores: {df_indicadores.shape}")

<h2 style="color:#A23B72; font-family:Georgia;">
🧹 2. Armonizar Nombres de Distritos
</h2>

Hay 3 distritos con nombres diferentes entre ambos datasets:
- `VILLA  ALTAGRACIA` (doble espacio) vs `VILLA ALTAGRACIA`
- `PERALVILLO` vs `ESPERALVILLO`
- `NEIBA` vs `NEYBA`

In [ ]:
DISTRITO_MAP = {
    "0404 - VILLA  ALTAGRACIA": "0404 - VILLA ALTAGRACIA",
    "1705 - PERALVILLO": "1705 - ESPERALVILLO",
    "1801 - NEIBA": "1801 - NEYBA",
}

df_centros["distrito"] = df_centros["distrito"].replace(DISTRITO_MAP)

# Verificar
distritos_c = set(df_centros["distrito"].unique())
distritos_i = set(df_indicadores[df_indicadores["tipo"] == "distrito"]["regional_distrito"].unique())
print(f"Distritos centros: {len(distritos_c)}")
print(f"Distritos indicadores: {len(distritos_i)}")
print(f"Intersección: {len(distritos_c & distritos_i)}")
print(f"Sin match: {distritos_c.symmetric_difference(distritos_i)}")

<h2 style="color:#A23B72; font-family:Georgia;">
🔗 3. Merge: Centros + Indicadores
</h2>

Cruzamos a nivel **distrito** (más granular que regional). Usamos los indicadores del último año disponible (2024-2025) como referencia principal, con datos históricos para calcular tendencias.

In [ ]:
COLS_INDICADOR = [
    "abandono_inicial", "promovido_inicial", "reprobado_inicial",
    "abandono_primario", "promovido_primario", "reprobado_primario",
    "abandono_secundario", "promovido_secundario", "reprobado_secundario",
]

# Indicadores a nivel distrito, último año
año_col = [c for c in df_indicadores.columns if "lectivo" in c][0]
ind_distrito_actual = df_indicadores[
    (df_indicadores["tipo"] == "distrito") &
    (df_indicadores[año_col] == "2024-2025")
][["regional_distrito"] + COLS_INDICADOR].copy()

ind_distrito_actual = ind_distrito_actual.rename(columns={"regional_distrito": "distrito"})

# Merge
df = df_centros.merge(ind_distrito_actual, on="distrito", how="left")

print(f"Shape después del merge: {df.shape}")
print(f"Centros con indicadores: {df['abandono_secundario'].notna().sum()} / {len(df)}")
print(f"Centros sin indicadores: {df['abandono_secundario'].isna().sum()}")
df.head()

<h2 style="color:#A23B72; font-family:Georgia;">
🏷️ 4. Crear Flags de Niveles Ofrecidos
</h2>

Cada centro puede ofrecer uno o varios niveles (ej. `INICIAL - PRIMARIO - SECUNDARIO`). Creamos columnas binarias para indicar qué niveles ofrece.

In [ ]:
df["tiene_inicial"] = df["nivel"].str.contains("INICIAL").astype(int)
df["tiene_primario"] = df["nivel"].str.contains("PRIMARIO").astype(int)
df["tiene_secundario"] = df["nivel"].str.contains("SECUNDARIO").astype(int)
df["tiene_adultos"] = df["nivel"].str.contains("ADULTOS|PREPARA").astype(int)
df["num_niveles"] = df["tiene_inicial"] + df["tiene_primario"] + df["tiene_secundario"] + df["tiene_adultos"]

# Sector como variable numérica
df["es_publico"] = (df["sector"] == "PUBLICO").astype(int)

print("Distribución de niveles ofrecidos:")
print(df[["tiene_inicial", "tiene_primario", "tiene_secundario", "tiene_adultos"]].sum())
print(f"\nSector público: {df['es_publico'].sum()} / {len(df)}")

<h2 style="color:#A23B72; font-family:Georgia;">
📐 5. Asignar Tasa de Abandono Relevante
</h2>

Cada centro recibe la tasa de abandono máxima entre los niveles que ofrece. Un centro que ofrece Primario y Secundario toma el mayor abandono de esos dos niveles como indicador de riesgo.

In [ ]:
def abandono_relevante(row):
    tasas = []
    if row["tiene_inicial"] and pd.notna(row["abandono_inicial"]):
        tasas.append(row["abandono_inicial"])
    if row["tiene_primario"] and pd.notna(row["abandono_primario"]):
        tasas.append(row["abandono_primario"])
    if row["tiene_secundario"] and pd.notna(row["abandono_secundario"]):
        tasas.append(row["abandono_secundario"])
    return max(tasas) if tasas else np.nan

df["abandono_max"] = df.apply(abandono_relevante, axis=1)
df["abandono_promedio"] = df[["abandono_inicial", "abandono_primario", "abandono_secundario"]].mean(axis=1)

print(f"abandono_max:\n{df['abandono_max'].describe().round(2)}")
print(f"\nabandono_promedio:\n{df['abandono_promedio'].describe().round(2)}")